# 03 - The RAG pipeline, step by step

**Question this notebook answers:** what exactly happens between a user's question and the answer?

```
01 Data + KB  ->  02 Build index  ->  03 RAG pipeline  ->  04 Evaluation
                                        (you are here)
```

```
question -> embed -> FAISS top 20 -> rerank, keep top 5 -> enough evidence?
                                                              no  -> abstain (no LLM call)
                                                              yes -> grounded prompt -> one LLM call
                                                                     -> answer + sources + disclaimer
```

Part 1 does every step by hand so you can see what goes in and what comes out.
Part 2 builds the same thing from the package code (`src/pipeline.py`), shows it returns the same passages, and runs a few questions through it.
The API (`app.py`) and the evaluation use that same package code.

**What the pipeline deliberately does not do:** no query classification or routing, no second retrieval pass, no retry with a looser prompt, and no fallback to the model's own knowledge. If the sources are not good enough, it says so.

In [1]:
import sys
from pathlib import Path

import torch
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.generator import build_messages, generate, make_client
from src.kb import load_encoder, load_index
from src.pipeline import RAGPipeline, format_result
from src.retriever import Retriever, load_reranker

index, chunks = load_index()
encoder = load_encoder()        # embeds the question (same model as the chunks)
reranker = load_reranker()      # cross-encoder for reranking
client = make_client()          # Groq, through the OpenAI SDK
threshold, calibrated = config.load_threshold()

print(f"{index.ntotal:,} chunks | abstain below a rerank score of {threshold} (calibrated: {calibrated})")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

31,673 chunks | abstain below a rerank score of 7.393619537353516 (calibrated: True)


## Part 1 - one question, step by step

In [2]:
question = "What are the symptoms of type 2 diabetes?"

### Step 1 - embed the question and search FAISS

The question becomes a vector with the same model used for the chunks. FAISS returns the 20 chunks whose vectors are closest.
This is fast and finds chunks about the same thing even with different wording, but the ranking is rough: question and chunk were embedded separately and are only compared at the end.

In [3]:
# Embed the question and take the top 20 by FAISS (dense search only)
query_vector = encoder.encode([question], normalize_embeddings=True).astype("float32")
print("query vector:", query_vector.shape)

scores, ids = index.search(query_vector, config.RETRIEVE_K)
candidates = chunks.iloc[ids[0]].copy()
candidates["faiss_score"] = scores[0]
candidates["faiss_rank"] = range(1, len(candidates) + 1)
candidates[["faiss_rank", "faiss_score", "source", "title"]].head(10)

query vector: (1, 1024)


,faiss_rank,faiss_score,source,title
4766,1,0.770674,MedQuAD (NIHSeniorHealth),Diabetes
22791,2,0.769211,MedQuAD (NIDDK),Am I at Risk for Type 2 Diabetes? Taking Steps...
29650,3,0.764685,MedlinePlus,Diabetes
17717,4,0.760566,MedQuAD (MPlusHealthTopics),Diabetes Type 2
29676,5,0.758164,MedlinePlus,Diabetes Type 2
25661,6,0.756526,MedQuAD (NIDDK),Your Guide to Diabetes: Type 1 and Type 2
29675,7,0.754919,MedlinePlus,Diabetes Type 2
26396,8,0.753959,MedQuAD (NIDDK),I Can Lower My Risk for Type 2 Diabetes: A Gui...
17716,9,0.711122,MedQuAD (MPlusHealthTopics),Diabetes Type 2
7889,10,0.705577,MedQuAD (GARD),"Lipodystrophy, familial partial, type 2"


### Step 2 - rerank

The cross-encoder reads the question and one chunk **together** and scores how well the chunk answers the question. It is more accurate but slower, so it only sees these 20 candidates.
The best 5 are kept. Compare their new order with `faiss_rank`.

In [4]:
# Rerank with the cross-encoder and keep the top 5
pairs = [(question, text) for text in candidates["text"]]
candidates["rerank_score"] = reranker.predict(pairs)

top = candidates.sort_values("rerank_score", ascending=False).head(config.CONTEXT_K)
top[["rerank_score", "faiss_rank", "source", "title"]]

,rerank_score,faiss_rank,source,title
22791,9.748358,2,MedQuAD (NIDDK),Am I at Risk for Type 2 Diabetes? Taking Steps...
4766,9.590372,1,MedQuAD (NIHSeniorHealth),Diabetes
25661,9.536401,6,MedQuAD (NIDDK),Your Guide to Diabetes: Type 1 and Type 2
17717,8.981016,4,MedQuAD (MPlusHealthTopics),Diabetes Type 2
29676,8.882860,5,MedlinePlus,Diabetes Type 2


### Step 3 - is there enough evidence?

Retrieval always returns *something*, even for a question the sources know nothing about. The rerank score of the best passage decides whether to answer.
The reranker outputs raw logits: above 0 roughly means "more likely relevant than not". The threshold comes from `src/config.py` until notebook 04 calibrates it on the dev questions.

In [5]:
# Evidence check: compare the best rerank score to the threshold
best = top["rerank_score"].iloc[0]
print(f"best rerank score: {best:.2f} | threshold: {threshold} | decision: {'answer' if best >= threshold else 'abstain'}")

best rerank score: 9.75 | threshold: 7.393619537353516 | decision: answer


### Step 4 - grounded prompt

The passages are numbered so the model can cite them as `[1]`, `[2]`. The system prompt tells it to use **only** these passages, to say what they do not cover, and not to give personal medical advice.

In [6]:
# Build the grounded prompt: system rules + numbered evidence passages
passages = top.to_dict("records")
system_message, user_message = build_messages(question, passages)
print(system_message["content"])
print("=" * 80)
print(user_message["content"][:2500])

You are an educational health information assistant. You are not a doctor and you do not give medical advice.

Answer the question using ONLY the numbered evidence passages. Do not use outside medical knowledge, even if you know the answer, and do not fill gaps from memory.

Structure the answer with short bold section titles when the passages contain information for them. For a condition, the usual sections are:

**Overview** - what it is, in one or two sentences.
**Symptoms** - what people notice.
**Causes** - why it happens or who is at risk.
**Diagnosis** - how doctors test for it.
**Treatment** - how it is usually managed.
**Prevention** - how to reduce the risk.
**When to see a doctor** - warning signs to act on.

Only include a section when the passages actually have information for it. Do not invent sections and do not pad empty ones. If the question is narrow (for example "what causes X?"), answer it directly and skip the overview.

Rules:
1. Cite the passages you used, like [

### Step 5 - one LLM call

One call, temperature 0. If the model returns nothing, `generate` raises an error instead of returning a blank answer.

In [7]:
# One LLM call; generate() raises if the model returns nothing
print(generate(client, question, passages))

**Symptoms**

• Increased thirst (very thirsty) – [1][2][3][4][5]  
• Frequent/increased urination – [1][2][3][4][5]  
• Increased hunger (very hungry) – [1][2][3][4][5]  
• Fatigue or feeling very tired – [1][2][3][5]  
• Unexplained weight loss (losing weight without trying) – [1][2][3][4][5]  
• Blurred vision (blurry eyesight) – [1][2][3][4][5]  
• Numbness, tingling, or “pins‑and‑needles” sensations in the feet or hands – [1][2][3][5]  
• Sores that heal slowly or do not heal – [1][2][3][4][5]  
• Dry, itchy skin – [2][3]  

These are the signs and symptoms of type 2 diabetes reported in the provided passages. If you notice any of these, consider speaking with a healthcare professional for evaluation.


### When the evidence is not good enough

Same steps, on a question about a condition that does not exist. The answer is not "ask the LLM anyway": the pipeline stops at step 3.

In [8]:
# Same idea on a question the sources cannot answer
retriever = Retriever(index, chunks, encoder, reranker)

invented = "What are the symptoms of Zelvarin-Moss syndrome?"
found = retriever.retrieve(invented)
print(f"best rerank score: {found[0]['rerank_score']:.2f} | threshold: {threshold} | decision: {'answer' if found[0]['rerank_score'] >= threshold else 'abstain'}")
print("best chunk anyway:", found[0]["source"], "-", found[0]["title"])

best rerank score: 5.32 | threshold: 7.393619537353516 | decision: abstain
best chunk anyway: MedQuAD (GARD) - Gorlin Chaudhry Moss syndrome


## Part 2 - the same thing as one object

`RAGPipeline` (`src/pipeline.py`) runs steps 1-5 and returns a dictionary. First, a check that it retrieves the same 5 passages as the manual steps:

In [9]:
# RAGPipeline should return the same top-5 chunks as the manual steps
pipeline = RAGPipeline(retriever, client, threshold, calibrated)

assert {p["chunk_id"] for p in retriever.retrieve(question)} == set(top["chunk_id"])
print("same top-5 chunks as the manual steps")

same top-5 chunks as the manual steps


`format_result` prints the answer, the numbered sources (only the passages that were in the prompt) and the disclaimer.
The questions below show different behaviours: a normal one, a personal-advice one (should abstain), a treatment-overview question, an invented syndrome (should abstain), a possible emergency, and a request for a medicine dose (should abstain, because medicines are out of scope).
Read each answer critically: do the `[n]` citations match the sources, and does the answer stay inside what the passages say?

In [10]:
# Different behaviours: normal, personal advice, emergency, out-of-scope dose
demo_questions = [
    "What are the symptoms of type 2 diabetes?",
    "My 3 year old has a fever and is very sleepy. What should I do?",
    "How is high blood pressure usually treated?",
    "What are the symptoms of Zelvarin-Moss syndrome?",
    "I have crushing chest pain that spreads to my left arm. What should I do?",
    "How many mg of ibuprofen should I give my 4 year old?",
]

for q in demo_questions:
    result = pipeline.answer(q)
    print("=" * 80)
    print("Q:", q)

    score = result.get("top_score")
    score_str = f"{score:.2f}" if score is not None else "n/a (short-circuited)"
    latency = result.get("latency_ms") or 0

    print(f"abstained: {result['abstained']} | best rerank score: {score_str} | {latency / 1000:.1f} s\n")
    print(format_result(result))

Q: What are the symptoms of type 2 diabetes?
abstained: False | best rerank score: 9.75 | 2.2 s

**Symptoms**

• Increased thirst (very thirsty) – [1][2][3][4][5]  
• Frequent/increased urination – [1][2][3][4][5]  
• Increased hunger (very hungry) – [1][2][3][4][5]  
• Fatigue or feeling very tired – [1][2][3][5]  
• Unexplained weight loss (losing weight without trying) – [1][2][3][4][5]  
• Blurred or blurry vision – [1][2][3][4][5]  
• Numbness, tingling, or “pins‑and‑needles” sensations in the feet or hands – [1][2][3][5]  
• Sores that heal slowly or do not heal – [1][2][3][4][5]  
• Dry, itchy skin – [2][3]   (mentioned in these sources)

Sources:
1. MedQuAD (NIDDK) - Am I at Risk for Type 2 Diabetes? Taking Steps to Lower Your Risk of Getting Diabetes
   http://www.niddk.nih.gov/health-information/health-topics/Diabetes/type-2-diabetes-taking-steps-lower-your-risk-diabetes/Pages/index.aspx
2. MedQuAD (NIHSeniorHealth) - Diabetes
   http://nihseniorhealth.gov/diabetes/toc.html
3

## Where the code lives

| Step | File | Function |
|---|---|---|
| Embed + FAISS top 20 | `src/retriever.py` | `Retriever.search` |
| Rerank, keep top 5 | `src/retriever.py` | `Retriever.rerank` |
| Evidence check, orchestration | `src/pipeline.py` | `RAGPipeline.answer` |
| Grounded prompt | `src/generator.py` | `SYSTEM_PROMPT`, `build_messages` |
| One LLM call | `src/generator.py` | `generate` |
| Settings (models, k, threshold) | `src/config.py` | |
| HTTP API | `app.py` | `POST /query`, `GET /health` |

**Next:** `04_evaluation.ipynb` measures retrieval, groundedness and safety behaviour against an LLM-only baseline, and calibrates the threshold used in step 3.